# Voz do consumidor: análise de avaliações de produtos com PLN

**Projeto de portfólio de Ciência de Dados** • Dataset B2W-Reviews01

**Objetivo:** compreender o comportamento das avaliações de produtos em português, quantificar divergências entre nota e recomendação e estudar a previsibilidade de avaliações favoráveis/desfavoráveis com representação TF-IDF.

**Dados:** 132.373 avaliações, 14 atributos, entre eles `review_text`, `overall_rating`, `recommend_to_a_friend`, `product_brand`. Estudo de caso: Samsung.

**Protocolo:** auditoria → análise exploratória → corpus filtrado → TF-IDF ajustado apenas no treino → regressão logística → avaliação no teste → interpretação lexical.

A classificação usa **rótulos derivados da nota**, não sentimentos anotados por humanos. Nota 3 é excluída apenas do modelo binário. Trabalho de apoio: relatório acadêmico e notebook previamente fornecidos, usados exclusivamente como subsídios conceituais. Este é um projeto analítico independente.

## 1. Obtenção da base e execução

O arquivo CSV **não foi republicado** porque contém atributos e identificadores de avaliadores. Obtenha `B2W-Reviews01.csv` na [pasta do projeto no Google Drive](https://drive.google.com/drive/folders/1VEffeCe6EGi0oVfsuPIkHrB5astVSv_K) e faça upload quando solicitado. Os resultados apresentados no relatório são de uma execução efetiva com essa base; a execução em outra versão deve recalcular tudo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay)
SEED = 42
candidatos = [Path('/content/B2W-Reviews01.csv'), Path('/content/Cópia de B2W-Reviews01.csv'), Path('B2W-Reviews01.csv')]
arquivo = next((p for p in candidatos if p.exists()), None)
if arquivo is None:
    from google.colab import files
    enviados = files.upload()
    candidatos = [Path(p) for p in enviados if p.lower().endswith('.csv')]
    if not candidatos: raise FileNotFoundError('Envie B2W-Reviews01.csv.')
    arquivo = candidatos[0]
df = pd.read_csv(arquivo, low_memory=False)
print(df.shape)
display(df.head(3)[['review_text','overall_rating','recommend_to_a_friend']])

## 2. Diagnóstico de qualidade

A base contém campos faltantes, especialmente para a marca do produto. Não imputamos marcas ou textos inexistentes, e não confundimos avaliações diferentes que tenham respostas textuais semelhantes. No modelo, eliminamos textos **exatamente duplicados** para reduzir vazamento entre treino e teste.

In [ ]:
campos = ['review_text','review_title','overall_rating','recommend_to_a_friend','product_brand']
auditoria = pd.DataFrame({'ausentes':df[campos].isna().sum(),
                         'percentual (%)':(100*df[campos].isna().mean()).round(2)})
display(auditoria.sort_values('percentual (%)',ascending=False))
print('Notas:',df.overall_rating.value_counts().sort_index().to_dict())
print('Textos repetidos:',df.review_text.dropna().duplicated().sum())
print('Categorias de recomendação:',df.recommend_to_a_friend.value_counts(dropna=False).to_dict())

## 3. Padrões de satisfação

As notas e as recomendações são indicadores diferentes: uma avaliação de nota baixa acompanhada de recomendação pode representar ambivalência ou problemas de preenchimento, sem que seja justificável descartar o registro automaticamente.

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
freq=df.overall_rating.value_counts().reindex(range(1,6),fill_value=0)
freq.plot.bar(ax=ax,color='#4263a6')
ax.set(xlabel='Nota',ylabel='Avaliações',title='Distribuição das notas')
ax.tick_params(axis='x',rotation=0); plt.tight_layout();plt.show()
validos=df[df.overall_rating.isin([1,2,4,5]) & df.recommend_to_a_friend.isin(['Yes','No'])]
divergente=((validos.overall_rating.le(2)&validos.recommend_to_a_friend.eq('Yes')) |
            (validos.overall_rating.ge(4)&validos.recommend_to_a_friend.eq('No')))
print('Divergências nas notas extremas:',divergente.sum(),f'({divergente.mean():.2%})')
display(pd.crosstab(df.overall_rating,df.recommend_to_a_friend,normalize='index').round(3))

## 4. Preparação do corpus Samsung

O filtro considera a marca padronizada por caixa e espaços. São mantidas apenas avaliações com texto de pelo menos 15 caracteres e notas 1, 2, 4 e 5. A variável-alvo vale 1 (notas 4–5) ou 0 (notas 1–2); excluímos cópias textuais exatas na modelagem. Essa seleção privilegia avaliações extremas e **não representa toda a experiência do consumidor**.

In [ ]:
base = df.copy()
base['brand_norm'] = base.product_brand.fillna('').astype(str).str.strip().str.casefold()
samsung = base.loc[base.brand_norm.eq('samsung')].copy()
samsung['texto'] = samsung.review_text.fillna('').astype(str).str.strip()
corpus = samsung.loc[(samsung.texto.str.len()>=15) & samsung.overall_rating.isin([1,2,4,5])].copy()
corpus = corpus.drop_duplicates(subset='texto').copy()
corpus['favoravel'] = corpus.overall_rating.ge(4).astype(int)
print('Samsung total:',len(samsung),'| Corpus modelagem:',len(corpus))
print('Proporção favorável:',round(corpus.favoravel.mean(),4))
assert corpus.favoravel.nunique()==2

## 5. Frequência lexical por classe

O gráfico permite verificar diferenças de frequência, sem pretender estabelecer causalidade. A frequência simples tende a destacar conectivos; na classificação, o TF-IDF atribui pesos discriminativos e mantém negações, úteis para interpretar elogios e críticas.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(15,5))
for ax,classe,rotulo in zip(axs,[0,1],['Desfavoráveis','Favoráveis']):
    cv=CountVectorizer(strip_accents='unicode',min_df=2,max_features=4000)
    mat=cv.fit_transform(corpus.loc[corpus.favoravel==classe,'texto'])
    freq=np.asarray(mat.sum(axis=0)).ravel()
    idx=np.argsort(freq)[-15:]
    ax.barh(cv.get_feature_names_out()[idx],freq[idx])
    ax.set_title(rotulo);ax.set_xlabel('Ocorrências')
plt.tight_layout();plt.show()

## 6. Classificação de avaliações com TF-IDF + regressão logística

Treino/teste estratificado (75/25) com semente fixa. O vocabulário, frequências documentais e ponderações do TF-IDF são aprendidos **somente no treino**. Os pesos de classe são balanceados para reduzir a dominância da classe majoritária. A avaliação utiliza métricas complementares e matriz de confusão.

In [ ]:
Xtr,Xte,ytr,yte = train_test_split(corpus.texto,corpus.favoravel,test_size=.25,random_state=SEED,stratify=corpus.favoravel)
tfidf = TfidfVectorizer(lowercase=True,strip_accents='unicode',min_df=3,max_df=.95,
                        max_features=12000,ngram_range=(1,2),sublinear_tf=True)
T=tfidf.fit_transform(Xtr);V=tfidf.transform(Xte)
modelo=LogisticRegression(max_iter=1200,class_weight='balanced',random_state=SEED)
modelo.fit(T,ytr);previsao=modelo.predict(V);prob=modelo.predict_proba(V)[:,1]
metricas={'Acurácia':accuracy_score(yte,previsao),
          'Acurácia balanceada':balanced_accuracy_score(yte,previsao),
          'F1 macro':f1_score(yte,previsao,average='macro'),
          'F1 positivo':f1_score(yte,previsao),
          'AUC ROC':roc_auc_score(yte,prob)}
display(pd.Series(metricas).round(4).to_frame('Valor'))
print(classification_report(yte,previsao,target_names=['Desfavorável','Favorável'],digits=3))
ConfusionMatrixDisplay.from_predictions(yte,previsao,display_labels=['Desfavorável','Favorável'],cmap='Blues')
plt.show()

## 7. Interpretação de termos

Os coeficientes identificam palavras e bigramas associados à previsão de cada classe. A associação não permite afirmar que a palavra **causou** a nota. Pode haver efeito da marca, categoria do produto, ironia, contexto e correlação entre termos.

In [ ]:
termos=np.asarray(tfidf.get_feature_names_out());coefs=modelo.coef_[0]
fig,axs=plt.subplots(1,2,figsize=(15,6))
for ax,ids,titulo in [(axs[0],np.argsort(coefs)[:18],'Associação desfavorável'),
                      (axs[1],np.argsort(coefs)[-18:],'Associação favorável')]:
    ids=ids[np.argsort(coefs[ids])]
    ax.barh(termos[ids],coefs[ids]);ax.set(title=titulo,xlabel='Coeficiente')
plt.tight_layout();plt.show()

## 8. Relatório executivo calculado

O texto é preenchido pelos resultados obtidos na sessão atual, não por métricas inventadas ou copiadas. Dessa forma, se o corpus mudar, o relatório também muda.

In [ ]:
from IPython.display import display, Markdown
resumo=(f'Foram examinadas {len(df):,} avaliações, com {len(samsung):,} avaliações '
        f'da Samsung identificadas. O conjunto binário final contém {len(corpus):,} textos '
        f'({corpus.favoravel.mean():.1%} favoráveis). O teste reservado compreende '
        f'{len(Xte):,} avaliações. Acurácia balanceada: {metricas["Acurácia balanceada"]:.3f}; '
        f'F1 macro: {metricas["F1 macro"]:.3f}; AUC ROC: {metricas["AUC ROC"]:.3f}. '
        'Os indicadores se referem somente a essa partição e não demonstram '
        'generalização para outros períodos ou marcas.')
display(Markdown('### Síntese dos resultados\n\n'+resumo))

## 9. Resultados verificados na execução de referência

O experimento foi executado com **132.373 registros**. Havia **6.015 avaliações Samsung** antes dos filtros; **5.169 textos distintos** foram utilizados na classificação, sendo **3.876 no treino** e **1.293 no teste**. A proporção de notas favoráveis no conjunto modelado foi de **87,83%**.

| Métrica no teste | Valor |
|---|---:|
| Acurácia | 0,9389 |
| Acurácia balanceada | 0,9241 |
| F1 macro | 0,8734 |
| F1 classe favorável | 0,9645 |
| AUC ROC | 0,9722 |

Matriz de confusão: **[[142, 15], [64, 1072]]** (linhas = classe real 0/1; colunas = classe predita 0/1). A interpretação deve considerar o desbalanceamento de classes e o fato de que a seleção excluiu a nota 3. São resultados da mesma base e protocolo descritos no código; não constituem validação externa.

## 10. Conclusões e limitações

A modelagem demonstra que, nesse conjunto e protocolo, as avaliações textuais contêm informação útil para inferir notas extremas. A investigação combina uma leitura descritiva e um modelo linear interpretável; isso permite examinar padrões lexicais associados às classes, mas não afirmar causalidade.

**Limitações:** viés de quem publica avaliações, notas como proxy de sentimento, uma única partição aleatória, possíveis diferenças entre produtos e épocas, falta de validação externa e dependência da qualidade da marca cadastrada.

**Próximas etapas:** validação temporal e por produto; classificação multiclasse com notas 1 a 5; anotação humana; análise de aspectos (entrega, preço, qualidade); comparação com Sentence-BERT e outros modelos contextuais.

**Fonte dos dados:** B2W-Reviews01, disponibilizado na [pasta original do projeto](https://drive.google.com/drive/folders/1VEffeCe6EGi0oVfsuPIkHrB5astVSv_K). Os atributos dos avaliadores e o CSV completo não são publicados no GitHub.